# День 9: регуляризация для дообученного бейзлайна + вторая регуляризованная GRU (~2.5 ч)
Итог day8: тройка ft_swa + gru_full + reg128 — **0.65512** на лидерборде (+0.0009) и уложилась по времени (0.8x). Регуляризация сработала: reg128 0.66544 против 0.66370 у gru_full, reg192 0.66437 против 0.66259, и к концу обучения валидация больше не падает.

Но сильнейшая модель смеси — ft_swa (дообученный бейзлайн, 0.66679) — дообучалась **без** регуляризации. Сегодня:
1. **ft_reg** — тот же рецепт дообучения, что у ft_swa (веса организаторов, 6 эпох, lr 3e-4, потом усреднение эпох 2–5), плюс dropout 0.2, шум 0.05, weight decay 1e-2 (~1.2 ч).
2. **reg128b** — GRU-128 с регуляризацией сильнее (dropout 0.3, шум 0.1, wd 2e-2) и другим сидом (~1.2 ч; можно пропустить и вернуться позже). Покажет, есть ли смысл усиливать регуляризацию, и даст ещё одну модель для тройки.
3. Оценка всех моделей размера 128 и выбор лучшей пары или тройки (тройка 128-х моделей по времени проверена: 0.8x). Сборка — если лучше отправленной тройки (0.67036) хотя бы на 0.0003.

Перед запуском: **T4 GPU**; загрузите на Drive вместо `wunder/day2` папку `хакатон/day2` — изменились `model.py`, `train.py`.

## 1. GPU + Drive + код + numba как у скорера

In [2]:
!nvidia-smi --query-gpu=name --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')
W = '/content/drive/MyDrive/wunder'
!pip -q install "numba==0.67.0" onnx 2>&1 | grep -v "dependency resolver\|requires numba\|incompatible" || true
!rm -rf /content/day2 && cp -r $W/day2 /content/day2
%cd /content/day2
!grep -q "gru and baseline" train.py && echo "code: OK (regularization for the baseline architecture present)" || echo "OLD CODE on Drive - upload the new day2 folder"

Tesla T4
Mounted at /content/drive
/content/day2
code: OK (regularization for the baseline architecture present)


## 2. Данные: train.parquet и baseline.onnx из стартового пакета (~6 мин)

In [3]:
import glob
URL = 'https://files.wundernn.io/wnn_connectome_starterpack.tar.gz'
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y aria2 > /dev/null 2>&1
!aria2c -c -x 16 -s 16 -k 1M --file-allocation=none --summary-interval=30 --console-log-level=warn -d /content -o pack.tar.gz $URL
!mkdir -p /content/sp && tar -xzf /content/pack.tar.gz -C /content/sp --wildcards '*datasets/train.parquet' '*baseline/baseline.onnx' 2>&1 | grep -v LIBARCHIVE
!rm -f /content/pack.tar.gz
TRAIN_PQ = glob.glob('/content/sp/**/datasets/train.parquet', recursive=True)[0]
ONNX = glob.glob('/content/sp/**/baseline/baseline.onnx', recursive=True)[0]
print(TRAIN_PQ, ONNX, sep='\n')

 *** Download Progress Summary as of Thu Nov 12 13:45:59 2026 *** 
=
[#e6d2ef 5.8GiB/31GiB(18%) CN:16 DL:224MiB ETA:1m56s]
FILE: /content/pack.tar.gz
-

 *** Download Progress Summary as of Thu Nov 12 13:46:29 2026 *** 
=
[#e6d2ef 12GiB/31GiB(39%) CN:16 DL:209MiB ETA:1m32s]
FILE: /content/pack.tar.gz
-

 *** Download Progress Summary as of Thu Nov 12 13:47:00 2026 *** 
=
[#e6d2ef 18GiB/31GiB(58%) CN:16 DL:235MiB ETA:57s]
FILE: /content/pack.tar.gz
-

 *** Download Progress Summary as of Thu Nov 12 13:47:30 2026 *** 
=
[#e6d2ef 24GiB/31GiB(78%) CN:16 DL:150MiB ETA:46s]
FILE: /content/pack.tar.gz
-

 *** Download Progress Summary as of Thu Nov 12 13:48:01 2026 *** 
=
[#e6d2ef 29GiB/31GiB(93%) CN:16 DL:205MiB ETA:9s]
FILE: /content/pack.tar.gz
-


Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
e6d2ef|OK  |   198MiB/s|/content/pack.tar.gz

Status Legend:
(OK):download completed.
/content/sp/wnn_connectome_s

## 3. Валидация — те же 600 последовательностей `valid_sub` (~1.5 мин)

In [4]:
!python prepare_data.py --src $W/colab_data/valid_sub.parquet --out /content/data/valid

100/600 seqs  10s  eta 52s
200/600 seqs  21s  eta 41s
300/600 seqs  29s  eta 29s
400/600 seqs  42s  eta 21s
500/600 seqs  52s  eta 10s
600/600 seqs  64s  eta 0s
{
 "src": "/content/drive/MyDrive/wunder/colab_data/valid_sub.parquet",
 "n_seqs": 600,
 "seq_len": 20000,
 "n_features": 112,
 "dtype": "float32",
 "has_is_scored": true,
 "nan_features": 0,
 "inf_features": 0,
 "nonfinite_targets": 0,
 "float16_overflow": 0,
 "scored_rows": 1144931
}


## 4. ft_reg: дообучение бейзлайна с регуляризацией (~1.2 ч)
Строка `VALID` перед первой эпохой — исходные веса организаторов. Если сессия оборвалась: ячейки 1–3, затем эта же ячейка с `--resume` в конце команды.

In [5]:
!python train.py --train-parquet $TRAIN_PQ --valid-dir /content/data/valid --out $W/runs/ft_reg \
    --arch baseline --init-onnx $ONNX --keep-epochs --epochs 6 --batch 32 --chunk 1000 --lr 3e-4 --mse-weight 0.1 \
    --dropout 0.2 --input-noise 0.05 --wd 1e-2 --loader-workers 4

device=cuda amp=True Tesla T4
train seqs 10607 from /content/sp/wnn_connectome_starterpack/datasets/train.parquet (parquet, read on the fly by 4 threads); valid seqs 600
initialised from /content/sp/wnn_connectome_starterpack/baseline/baseline.onnx
VALID (start = baseline weights) {"seqs": 600, "pred_std": [1.7675000429153442, 1.694100022315979], "WP_mask_std0.15": 0.66112, "WP_mask_std0.25": 0.66112, "WP_mask_std0.35": 0.66112, "WP_mask_std0.5": 0.66155, "WP_mask_std0.7": 0.65793, "WP_mask_std1.0": 0.64296, "WP_mask_std1.5": 0.61367, "WP_mask_raw": 0.60174, "WP_mask_best": 0.66155, "best_target_std": 0.5, "output_scale": [0.2828899919986725, 0.2951500117778778], "WP_all_at_best": 0.5027, "epoch": -1, "train_minutes": 0.0}
params: 192,258
ep 0 batch 10/332 loss 0.9974 lr 2.01e-04 0.27M rows/s data-wait 43% eta 12.8 min
ep 0 batch 20/332 loss 0.8754 lr 3.00e-04 0.33M rows/s data-wait 35% eta 10.2 min
ep 0 batch 30/332 loss 0.8552 lr 3.00e-04 0.35M rows/s data-wait 31% eta 9.1 min
ep 0 b

## 5. reg128b: GRU-128, регуляризация сильнее, другой сид (~1.2 ч; можно пропустить)

In [6]:
!python train.py --train-parquet $TRAIN_PQ --valid-dir /content/data/valid --out $W/runs/reg128b \
    --arch gru --hidden 128 --epochs 8 --batch 64 --chunk 1000 --lr 2e-3 --keep-epochs --seed 1 \
    --dropout 0.3 --input-noise 0.1 --wd 2e-2 --loader-workers 4

device=cuda amp=True Tesla T4
train seqs 10607 from /content/sp/wnn_connectome_starterpack/datasets/train.parquet (parquet, read on the fly by 4 threads); valid seqs 600
params: 221,986
ep 0 batch 10/166 loss 0.8946 lr 1.34e-03 0.40M rows/s data-wait 43% eta 8.3 min
ep 0 batch 20/166 loss 0.8137 lr 2.00e-03 0.43M rows/s data-wait 43% eta 7.3 min
ep 0 batch 30/166 loss 0.7767 lr 2.00e-03 0.44M rows/s data-wait 43% eta 6.6 min
ep 0 batch 40/166 loss 0.7553 lr 2.00e-03 0.44M rows/s data-wait 43% eta 6.1 min
ep 0 batch 50/166 loss 0.7489 lr 1.99e-03 0.45M rows/s data-wait 43% eta 5.6 min
ep 0 batch 60/166 loss 0.7404 lr 1.99e-03 0.45M rows/s data-wait 43% eta 5.1 min
ep 0 batch 70/166 loss 0.7387 lr 1.99e-03 0.45M rows/s data-wait 43% eta 4.6 min
ep 0 batch 80/166 loss 0.7429 lr 1.98e-03 0.45M rows/s data-wait 43% eta 4.1 min
ep 0 batch 90/166 loss 0.7470 lr 1.98e-03 0.45M rows/s data-wait 43% eta 3.6 min
ep 0 batch 100/166 loss 0.7348 lr 1.97e-03 0.45M rows/s data-wait 43% eta 3.1 min
ep 

## 6. Все модели размера 128 на одной валидации (~10 мин)

In [7]:
import os
def swa(run, ks):
    return '+'.join(f'{run}/epoch_{k}.pt' for k in ks)
CANDS = {
    'ft_swa':      swa(W + '/runs/ft_baseline_full', range(2, 6)),
    'ft_reg_swa':  swa(W + '/runs/ft_reg', range(2, 6)),
    'ft_reg_last': W + '/runs/ft_reg/epoch_5.pt',
    'gru_full':    W + '/runs/gru_full/best.pt',
    'reg128':      W + '/runs/reg128/best.pt',
    'reg128b':     W + '/runs/reg128b/best.pt',
}
FOUND = {k: v for k, v in CANDS.items() if all(os.path.exists(p) for p in v.split('+'))}
print('models:', list(FOUND), '| missing:', [k for k in CANDS if k not in FOUND])
MODELS = ' '.join(f'--model {k}={v}' for k, v in FOUND.items())
!python ensemble_eval.py --valid-dir /content/data/valid --out /content/preds9 $MODELS --report $W/day9.json

models: ['ft_swa', 'ft_reg_swa', 'ft_reg_last', 'gru_full', 'reg128', 'reg128b'] | missing: []
ft_swa         predicted (baseline, epoch 2, SWA of 4)
ft_reg_swa     predicted (baseline, epoch 2, SWA of 4)
ft_reg_last    predicted (baseline, epoch 5)
gru_full       predicted (gru, epoch 5)
reg128         predicted (gru, epoch 6)
reg128b        predicted (gru, epoch 6)

validation: 600 sequences, 1,144,931 scored rows. WP on the mask (and on all required rows):
  ft_swa         0.66679   (0.51096)
  ft_reg_swa     0.66476   (0.50889)
  ft_reg_last    0.66443   (0.50751)
  gru_full       0.66370   (0.50515)
  reg128         0.66544   (0.50999)
  reg128b        0.66552   (0.51174)

correlation of predictions on scored rows (lower = more to gain from blending):
                     ft_swa ft_reg_swa ft_reg_las   gru_full     reg128    reg128b
  ft_swa              1.000      0.989      0.989      0.963      0.968      0.968
  ft_reg_swa          0.989      1.000      1.000      0.966      0

## 7. Выбор сабмита
Все модели здесь размера 128, поэтому подходит любая пара (0.53x) или тройка (0.8x — так уже прошла отправленная). Сборка — если смесь лучше отправленной тройки ft_swa+gru_full+reg128 хотя бы на 0.0003.

In [8]:
import json
eb = json.load(open(W + '/day9.json'))['equal_blends']
SENT = 'ft_swa+gru_full+reg128'
cands = sorted(((v, k) for k, v in eb.items() if k.count('+') <= 2), reverse=True)
print(f'sent: {SENT} {eb[SENT]:.5f}')
for v, k in cands[:8]:
    print(f'  {k:36s} {v:.5f} ({v - eb[SENT]:+.5f})  time {"0.80x" if k.count("+") == 2 else "0.53x"}')
BEST_V, BEST = cands[0]
BUILD = BEST_V > eb[SENT] + 0.0003
OUT = W + '/submissions/ens9_' + BEST.replace('+', '_')
SPEC = ' '.join(f'--model "{n}={FOUND[n]}"' for n in BEST.split('+'))
REF = W + '/submissions/gru128x2_v1'
print('build:', BEST if BUILD else 'nothing - no candidate beats the sent ensemble by 0.0003')

sent: ft_swa+gru_full+reg128 0.67036
  ft_swa+gru_full+reg128               0.67036 (+0.00000)  time 0.80x
  ft_swa+gru_full+reg128b              0.67022 (-0.00013)  time 0.80x
  ft_swa+reg128+reg128b                0.66996 (-0.00040)  time 0.80x
  ft_swa+reg128                        0.66993 (-0.00043)  time 0.53x
  ft_swa+gru_full                      0.66987 (-0.00049)  time 0.53x
  ft_swa+ft_reg_swa+gru_full           0.66965 (-0.00071)  time 0.80x
  ft_swa+ft_reg_last+gru_full          0.66962 (-0.00074)  time 0.80x
  ft_swa+reg128b                       0.66960 (-0.00076)  time 0.53x
build: nothing - no candidate beats the sent ensemble by 0.0003


## 8. Сборка, проверка, скорость, скачивание (~10 мин)

In [1]:
if BUILD:
    !python build_ensemble.py --valid-dir /content/data/valid --out $OUT $SPEC
    !python check_submission.py {OUT}.zip --data $W/colab_data/valid_sub.parquet --seqs 3 --rows 20000 --ref-dir $REF
    !python bench_speed.py $REF $OUT --rows 5000 --repeats 5
    from google.colab import files
    files.download(OUT + '.zip')

NameError: name 'BUILD' is not defined

## Что прислать
Строки `VALID` из обоих обучений и вывод ячеек 6–8. Если zip собрался и проверка прошла — можно отправлять.